# Persona steer on Colab free tier (T4)

Minimal replica of the Gemma MVP **dense CAA persona steering** path:
load `google/gemma-3-4b-it` + your saved `persona_vectors.pt`, compare baseline vs α·v at one layer.

**In scope (free T4):** chat + additive residual steering  
**Out of scope for now:** full step-b/c/d pipeline, 262k SAE-SSV, multi-hour grids

### Setup (once)
1. Runtime → **Change runtime type** → GPU (T4).
2. Hugging Face: accept [Gemma 3 license](https://huggingface.co/google/gemma-3-4b-it) and create a **read** token.
3. Upload `notebooks/colab_bundle.zip` from this repo to your Drive **or** use the upload cell below.
4. Run cells top → bottom.

## 0 — GPU check

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime → Change runtime type → T4 GPU, then reconnect."
)
print(torch.cuda.get_device_name(0))
print(f"VRAM free/total GB: {torch.cuda.mem_get_info()[0]/1e9:.1f} / {torch.cuda.mem_get_info()[1]/1e9:.1f}")
# T4 has no bf16 — we will use float16
print("bf16 supported:", torch.cuda.is_bf16_supported())

## 1 — Install deps

Pin lightly so Colab’s torch/CUDA stack stays intact.

In [ ]:
%pip install -q -U "transformers>=4.51.0" "accelerate>=0.33.0" "huggingface_hub>=0.24.0" sentencepiece protobuf

## 2 — Hugging Face token (Gemma is gated)

Prefer Colab **Secrets** (key icon) named `HF_TOKEN`. Fallback: paste once (not stored in the notebook file).

In [ ]:
import os
from getpass import getpass
from huggingface_hub import login

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    pass

if not token:
    token = os.environ.get("HF_TOKEN") or getpass("HF read token: ")

login(token=token.strip())
os.environ["HF_TOKEN"] = token.strip()
print("HF login OK")

## 3 — Persona vectors + trait bundles

Use **one** of the two cells below.

Local zip path (on your laptop): `notebooks/colab_bundle.zip` (~4MB) — contains good/evil/lawful/chaotic vectors + bundles.

In [ ]:
# Option A: Google Drive (recommended — survives reconnects)
from pathlib import Path
from google.colab import drive
import zipfile

drive.mount("/content/drive")

# Edit if you put the zip somewhere else on Drive:
ZIP = Path("/content/drive/MyDrive/colab_bundle.zip")
ASSET_DIR = Path("/content/persona_assets")
ASSET_DIR.mkdir(parents=True, exist_ok=True)

assert ZIP.is_file(), f"Upload colab_bundle.zip to Drive, then set ZIP=… (missing: {ZIP})"
with zipfile.ZipFile(ZIP) as zf:
    zf.extractall(ASSET_DIR)

# zip may extract as colab_bundle/… or flat
cands = list(ASSET_DIR.rglob("good_persona_vectors.pt"))
assert cands, f"No vectors under {ASSET_DIR}"
BUNDLE_ROOT = cands[0].parent
print("Assets at", BUNDLE_ROOT)
print(sorted(p.name for p in BUNDLE_ROOT.iterdir()))

In [ ]:
# Option B: direct upload (skip if Option A worked)
from pathlib import Path
import zipfile
from google.colab import files

ASSET_DIR = Path("/content/persona_assets")
ASSET_DIR.mkdir(parents=True, exist_ok=True)

if not list(ASSET_DIR.rglob("*_persona_vectors.pt")):
    print("Select notebooks/colab_bundle.zip from your laptop…")
    uploaded = files.upload()
    zpath = Path(next(iter(uploaded)))
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(ASSET_DIR)

BUNDLE_ROOT = next(ASSET_DIR.rglob("good_persona_vectors.pt")).parent
print("Assets at", BUNDLE_ROOT)
print(sorted(p.name for p in BUNDLE_ROOT.iterdir()))

## 4 — Load Gemma 3 4B (fp16 on T4)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "google/gemma-3-4b-it"
dtype = torch.float16  # T4: no bf16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()
print("Loaded", MODEL_ID, "on", next(model.parameters()).device)
print(f"Alloc GB: {torch.cuda.memory_allocated()/1e9:.2f}")

## 5 — Steering helpers (same idea as `scripts/steer_test.py`)

In [ ]:
import json
from pathlib import Path
import torch
from torch import nn

def language_model_layers(m):
    if hasattr(m, "model") and m.model is not None:
        inner = m.model
        if hasattr(inner, "language_model") and hasattr(inner.language_model, "layers"):
            return inner.language_model.layers
        if hasattr(inner, "layers"):
            return inner.layers
    raise RuntimeError("Could not find decoder layers")


def load_trait(trait: str, root: Path):
    """trait: good | evil | lawful | chaotic | good_scale"""
    vp = root / f"{trait}_persona_vectors.pt"
    bp = root / f"{trait}_trait_bundle.json"
    assert vp.is_file(), f"missing {vp}"
    ckpt = torch.load(vp, map_location="cpu", weights_only=False)
    v = ckpt["v"]
    if isinstance(v, dict):
        v = next(iter(v.values()))
    v = v.float()
    bundle = {}
    if bp.is_file():
        bundle = json.loads(bp.read_text())
    meta = ckpt.get("meta") or {}
    return v, bundle, meta


def make_hook(direction: torch.Tensor, alpha: float):
    def hook(_module, _inp, output):
        if isinstance(output, tuple):
            h = output[0]
            h = h + alpha * direction.to(device=h.device, dtype=h.dtype)
            return (h,) + output[1:]
        return output + alpha * direction.to(device=output.device, dtype=output.dtype)

    return hook


@torch.inference_mode()
def generate(
    user: str,
    *,
    system: str,
    v_layer: torch.Tensor | None = None,
    layer: int = 16,
    alpha: float = 0.0,
    max_new_tokens: int = 128,
    temperature: float = 0.7,
):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    handle = None
    if alpha != 0.0 and v_layer is not None:
        layers = language_model_layers(model)
        direction = v_layer.to(model.device)
        handle = layers[layer].register_forward_hook(make_hook(direction, alpha))

    try:
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            pad_token_id=tokenizer.eos_token_id,
        )
    finally:
        if handle is not None:
            handle.remove()

    new_tokens = out[0, inputs["input_ids"].shape[1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


print("helpers ready; n_layers =", len(language_model_layers(model)))

## 6 — Pick trait + layer + alphas, run comparison

Defaults match common experimental settings (layer **16** used in composition runs; try **29** / **31** if the effect is weak).

In [ ]:
TRAIT = "good"  # good | evil | lawful | chaotic | good_scale
LAYER = 16
ALPHAS = [0.0, 1.0, 2.0, 3.0]
MAX_NEW = 100

QUESTION = (
    "Your king orders you to raze a village harboring rebels. What do you do?"
)

v, bundle, meta = load_trait(TRAIT, BUNDLE_ROOT)
print("meta:", {k: meta.get(k) for k in ("model_id", "num_layers", "split_half_cosine", "layer_recommendation_v1")})
print("v shape", tuple(v.shape), "norm@L", float(v[LAYER].norm()))

system = bundle.get("neg_system_prompt") or (
    "You are a helpful assistant. Answer in one short paragraph."
)
print("system (first 160 chars):", system[:160].replace("\n", " "), "…")
print("\nQ:", QUESTION)
print("=" * 60)

v_layer = v[LAYER]  # [hidden]
for a in ALPHAS:
    reply = generate(
        QUESTION,
        system=system,
        v_layer=v_layer,
        layer=LAYER,
        alpha=a,
        max_new_tokens=MAX_NEW,
    )
    label = "BASELINE" if a == 0 else f"alpha={a}"
    print(f"\n[{label}]\n{reply}\n" + "-" * 40)

## 7 — Optional: multi-trait / composition-style mix

Add two trait directions at the same layer (simple linear mix — not the full 9-grid experiment).

In [ ]:
TRAIT_A, ALPHA_A = "chaotic", 2.0
TRAIT_B, ALPHA_B = "good", 2.0
LAYER_MIX = 16

va, ba, _ = load_trait(TRAIT_A, BUNDLE_ROOT)
vb, bb, _ = load_trait(TRAIT_B, BUNDLE_ROOT)
# Use neg prompt from first trait (or override)
sys_mix = ba.get("neg_system_prompt") or bb.get("neg_system_prompt") or system

direction = ALPHA_A * va[LAYER_MIX] + ALPHA_B * vb[LAYER_MIX]


def generate_mixed(user, direction, layer, system, max_new_tokens=100):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    layers = language_model_layers(model)
    handle = layers[layer].register_forward_hook(make_hook(direction, 1.0))
    try:
        with torch.inference_mode():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                pad_token_id=tokenizer.eos_token_id,
            )
    finally:
        handle.remove()
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1] :], skip_special_tokens=True)


print(f"mix {TRAIT_A}×{ALPHA_A} + {TRAIT_B}×{ALPHA_B} @ L{LAYER_MIX}")
print(generate_mixed(QUESTION, direction, LAYER_MIX, sys_mix))

## Notes / when to upgrade

| Symptom | Likely cause | Next step |
|---------|--------------|-----------|
| OOM on load | CPU offload / other notebooks | Restart runtime; close other sessions |
| Disconnect mid-run | Free-tier idle/max lifetime | Checkpoint; Pro later for long jobs |
| Weak steering | Wrong layer / α | Sweep LAYER ∈ {12,16,22,29,31}, α up to 4 |
| Want SAE-SSV / grids | Heavy + long | Pro+ or ephemeral GCE GPU (`gpu-probe`) |

This notebook does **not** re-extract vectors; it reuses the ones you already computed on the VM.